# Chapter 2 — The Action Boundary

**Book alignment:** current Chapter 2 · internal demo `Stage 01`

The shared demo package calls this **Stage 01** internally. The notebook number follows the book chapter number; the internal stage number is one lower.

**Question this notebook isolates:** Can a proposal fail at a precise boundary stage without ever reaching the environment?


In [ ]:
from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "demo" / "agents-from-first-principles").exists():
            return candidate
    raise RuntimeError(
        "Run this notebook from a checkout containing demo/agents-from-first-principles"
    )


REPO_ROOT = find_repo_root(Path.cwd().resolve())
DEMO_ROOT = REPO_ROOT / "demo" / "agents-from-first-principles"
sys.path.insert(0, str(DEMO_ROOT))

from first_principles_agent import (
    AcceptanceStage,
    ActionAcceptanceBoundary,
    Agent,
    RepositoryEnvironment,
    Stage01Policy,
)
from first_principles_agent.actions import Action, RawProposal

## One boundary, five different failures

Use the real broken-parser repository as the authorization and precondition context.


In [ ]:
target = DEMO_ROOT / "examples" / "broken-parser"
boundary = ActionAcceptanceBoundary(target)

cases = {
    "malformed representation": RawProposal('{"action": "read_file"'),
    "bad schema": RawProposal('{"target": "parser.py"}'),
    "unsupported semantics": RawProposal('{"action": "delete_file"}'),
    "unauthorized target": RawProposal(
        '{"action": "read_file", "target": "../secret.txt"}'
    ),
    "missing precondition": RawProposal(
        '{"action": "read_file", "target": "missing.py"}'
    ),
    "accepted": RawProposal('{"action": "read_file", "target": "parser.py"}'),
}

results = {name: boundary.accept(proposal) for name, proposal in cases.items()}
[
    (name, result.accepted, result.stage.value, result.reason)
    for name, result in results.items()
]

In [ ]:
assert results["malformed representation"].stage == AcceptanceStage.REPRESENTATION
assert results["bad schema"].stage == AcceptanceStage.SCHEMA
assert results["unsupported semantics"].stage == AcceptanceStage.SEMANTICS
assert results["unauthorized target"].stage == AcceptanceStage.AUTHORIZATION
assert results["missing precondition"].stage == AcceptanceStage.PRECONDITIONS
assert results["accepted"].stage == AcceptanceStage.ACCEPTED
assert results["accepted"].action is not None

print("Five rejection/acceptance surfaces remain distinguishable.")

## Rejected really means not executed

The next environment raises immediately if it is called. If the authorization boundary works, the run should terminate before execution.


In [ ]:
class UnauthorizedPolicy:
    def choose_action(self, _state):
        return RawProposal('{"action": "read_file", "target": "../secret.txt"}')


class MustNotExecuteEnvironment:
    def execute(self, _action: Action):
        raise AssertionError("rejected proposal reached the environment")


blocked = Agent(
    policy=UnauthorizedPolicy(),
    environment=MustNotExecuteEnvironment(),
    acceptance=boundary,
).run("Read outside the repository")

blocked.termination_reason

In [ ]:
assert blocked.transitions == []
assert len(blocked.acceptance_attempts) == 1
assert blocked.acceptance_attempts[0].stage == AcceptanceStage.AUTHORIZATION
assert blocked.termination_reason.startswith("proposal rejected at authorization:")

print("Unauthorized proposal → zero executed transitions")

## Run the real Stage-01 agent

The control logic is still the same as Stage 00. The difference is that every choice is now emitted as raw JSON and must earn an executable typed `Action`.


In [ ]:
agent = Agent(
    policy=Stage01Policy(),
    environment=RepositoryEnvironment(target),
    acceptance=boundary,
)
state = agent.run("Fix pipe-delimited records")

(
    [
        (
            attempt.stage.value,
            attempt.accepted,
            attempt.action.kind.value if attempt.action else None,
        )
        for attempt in state.acceptance_attempts
    ],
    [transition.action.kind.value for transition in state.transitions],
    state.termination_reason,
)

In [ ]:
assert [attempt.stage for attempt in state.acceptance_attempts] == [
    AcceptanceStage.ACCEPTED,
    AcceptanceStage.ACCEPTED,
    AcceptanceStage.ACCEPTED,
]
assert [transition.action.kind.value for transition in state.transitions] == [
    "run_tests",
    "read_file",
]
assert state.termination_reason == (
    "inspection complete; repair capability has not been introduced yet"
)

## What we earned

Stage 01 establishes a hard architectural boundary:

> **The policy proposes. The runtime decides whether the proposal may become executable.**

Parsing, schema validity, semantic validity, authorization, and execution preconditions are now different failure surfaces. A proposal rejected at any one of them produces **no environment action**.

Notebook 03 / Chapter 3 keeps this boundary and adds **multiple complete proposals plus selection**, without yet turning the system into trajectory search.
